# NB09 — Ablation Training Runs

Trains **BanglaBERT only, seed=42** under 4 ablation conditions.
Results feed directly into Table 2 in NB07.

**Requires GPU. ~1 hr per condition (~4 hrs total).**

Run after NB05_v2 (uses same data splits and label encoders).

In [1]:
import os, json, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, accuracy_score, matthews_corrcoef
from collections import defaultdict

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')
os.makedirs('../outputs/ablations', exist_ok=True)


/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


## Load data + apply same label consolidation as NB05_v2

In [2]:
SPLITS_DIR = '../data/splits'
train_df = pd.read_csv(f'{SPLITS_DIR}/random_train.csv')
val_df   = pd.read_csv(f'{SPLITS_DIR}/random_val.csv')
test_df  = pd.read_csv(f'{SPLITS_DIR}/random_test.csv')

# ── Identical consolidation from NB05_v2 ──────────────────────────────
TYPE_MAP = {
    'none':'none','not bully':'none',
    'sexual':'sexual',
    'threat':'threat','threat,spam':'threat',
    'callToViolence':'threat','callToViolence_slander':'threat',
    'callToViolence_gender':'threat','callToViolence_religion':'threat',
    'callToViolence_religion_slander':'threat',
    'callToViolence_gender_religion_slander':'threat',
    'callToViolence_gender_slander':'threat',
    'religious,threat':'threat','sexual,threat':'threat',
    'sexual,religious,threat':'threat',
    'religious':'religious','Religious':'religious',
    'religion':'religious','religious,spam':'religious',
    'religion_slander':'religious',
    'gender_religion':'religious','gender_religion_slander':'religious',
    'sexual,religious':'religious',
    'gender':'gender','Gender':'gender','gender_slander':'gender',
    'Political':'political',
    'Personal Offense':'personal','Body Shaming':'personal',
    'Origin':'personal','slander':'personal','Misc':'personal',
    'Abusive/Violence':'abusive','troll':'abusive',
    'spam':'other',
}
PRIORITY = ['threat','sexual','religious','gender','political',
            'abusive','personal','other','none']

def consolidate_type(val):
    if not isinstance(val, str) or val.strip() == '': return 'none'
    val = val.strip()
    if val in TYPE_MAP: return TYPE_MAP[val]
    parts = [v.strip() for v in val.replace(',','|').replace(';','|').split('|')]
    candidates = [TYPE_MAP[p] for p in parts if p in TYPE_MAP]
    if candidates:
        for pc in PRIORITY:
            if pc in candidates: return pc
    for key, mapped in TYPE_MAP.items():
        if key.lower() in val.lower(): return mapped
    return 'other'

for df_ in [train_df, val_df, test_df]:
    df_['label_type'] = df_['label_type'].apply(consolidate_type)

print('Label consolidation done')
print(f'Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')


Label consolidation done
Train: 108,460  Val: 13,557  Test: 13,558


## Shared model classes (identical to NB05_v2)

In [3]:
class CyberBullyDataset(Dataset):
    def __init__(self, df, tokenizer, max_length, active_tasks, label_encoders, text_col='text_clean'):
        self.texts = df[text_col].fillna('').tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.labels = {}
        for task_name, task_cfg in active_tasks.items():
            col = task_cfg['col']
            enc = label_encoders[task_name]
            self.labels[task_name] = [
                enc.get(str(v) if not pd.isna(v) else 'unknown', -1)
                for v in df[col].fillna('unknown')
            ]
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_length,
                              padding='max_length', truncation=True, return_tensors='pt')
        item = {k: v.squeeze(0) for k, v in enc.items()}
        for t in self.labels:
            item[f'label_{t}'] = torch.tensor(self.labels[t][idx], dtype=torch.long)
        return item

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, weight=self.weight, reduction='none')
        pt = torch.exp(-ce)
        return (((1-pt)**self.gamma) * ce).mean()

class MultiTaskTransformer(nn.Module):
    def __init__(self, model_name, active_tasks, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        h = self.encoder.config.hidden_size
        self.heads    = nn.ModuleDict()
        self.dropouts = nn.ModuleDict()
        for t, cfg in active_tasks.items():
            self.dropouts[t] = nn.Dropout(dropout)
            self.heads[t]    = nn.Linear(h, cfg['num_classes'])
    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kw = {'input_ids': input_ids, 'attention_mask': attention_mask}
        if token_type_ids is not None: kw['token_type_ids'] = token_type_ids
        cls = self.encoder(**kw).last_hidden_state[:, 0, :]
        return {t: self.heads[t](self.dropouts[t](cls)) for t in self.heads}

def get_layerwise_params(model, base_lr, decay, wd):
    no_decay = ['bias', 'LayerNorm.weight', 'LayerNorm.bias']
    n_layers = sum(1 for n,_ in model.encoder.named_parameters()
                   if any(p.isdigit() for p in n.split('.'))) // 10 or 12
    groups = []
    for name, param in model.encoder.named_parameters():
        if not param.requires_grad: continue
        layer_num = next((int(p) for p in name.split('.') if p.isdigit()), 0)
        lr = base_lr * (decay ** (n_layers - layer_num - 1))
        groups.append({'params':[param],'lr':lr,
                        'weight_decay':0.0 if any(nd in name for nd in no_decay) else wd})
    for name, param in model.heads.named_parameters():
        if param.requires_grad:
            groups.append({'params':[param],'lr':base_lr,
                            'weight_decay':0.0 if any(nd in name for nd in no_decay) else wd})
    return groups

@torch.no_grad()
def evaluate_binary(model, dataloader):
    model.eval()
    all_preds, all_labels = [], []
    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = model(input_ids=batch['input_ids'],
                       attention_mask=batch['attention_mask'],
                       token_type_ids=batch.get('token_type_ids'))
        preds  = logits['binary'].argmax(dim=-1).cpu().numpy()
        labels = batch['label_binary'].cpu().numpy()
        all_preds.extend(preds); all_labels.extend(labels)
    y_true, y_pred = np.array(all_labels), np.array(all_preds)
    return {
        'accuracy':    round(accuracy_score(y_true, y_pred), 4),
        'macro_f1':    round(f1_score(y_true, y_pred, average='macro'), 4),
        'weighted_f1': round(f1_score(y_true, y_pred, average='weighted'), 4),
        'mcc':         round(matthews_corrcoef(y_true, y_pred), 4),
    }

print('Classes defined')


Classes defined


## Ablation Conditions

| Tag | What changes |
|---|---|
| `full_multitask` | Full NB05_v2 system (reference, 1 seed) |
| `binary_only` | Remove abuse_type head entirely |
| `no_focal` | Replace FocalLoss with standard CrossEntropy |
| `no_lrdecay` | Uniform LR across all layers |
| `no_preprocessing` | Use raw `text` column instead of `text_clean` |

In [4]:
BASE_MODEL = 'csebuetnlp/banglabert'
BASE_MODEL_KEY = 'banglabert'
SEED = 42
BASE_CONFIG = {
    'max_length': 128, 'batch_size': 32, 'epochs': 10,
    'learning_rate': 2e-5, 'weight_decay': 0.01, 'warmup_ratio': 0.1,
    'lr_decay_factor': 0.95, 'focal_gamma': 2.0, 'patience': 3,
    'use_fp16': torch.cuda.is_available(),
}

# ── Build label encoders for binary + consolidated type ───────────────
TASK_CONFIG_FULL = {
    'binary':     {'col':'label_binary',  'num_classes':2,    'loss_weight':1.0},
    'abuse_type': {'col':'label_type',    'num_classes':None, 'loss_weight':0.5},
}
TASK_CONFIG_BINARY_ONLY = {
    'binary': {'col':'label_binary', 'num_classes':2, 'loss_weight':1.0},
}

def build_label_encoders(task_cfg):
    encoders = {}
    for t, cfg in task_cfg.items():
        col = cfg['col']
        if col in train_df.columns:
            vals = pd.concat([train_df[col], val_df[col], test_df[col]]).dropna().unique()
            encoders[t] = {v: i for i, v in enumerate(sorted(vals))}
            cfg['num_classes'] = len(encoders[t])
            print(f"  Task '{t}': {cfg['num_classes']} classes")
    return encoders

import copy
full_cfg  = copy.deepcopy(TASK_CONFIG_FULL)
bin_cfg   = copy.deepcopy(TASK_CONFIG_BINARY_ONLY)

print('Full task encoders:')
label_enc_full   = build_label_encoders(full_cfg)
print('Binary-only encoders:')
label_enc_binary = build_label_encoders(bin_cfg)


Full task encoders:
  Task 'binary': 2 classes
  Task 'abuse_type': 9 classes
Binary-only encoders:
  Task 'binary': 2 classes


## Core training function

In [5]:
def run_ablation(tag, active_tasks, label_encoders, config,
                 use_focal=True, use_lrdecay=True, text_col='text_clean'):
    print(f'\n{"="*60}')
    print(f'ABLATION: {tag}  |  {BASE_MODEL_KEY}  |  seed={SEED}')
    print(f'{"="*60}')

    torch.manual_seed(SEED); np.random.seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    mk = {'active_tasks': active_tasks, 'label_encoders': label_encoders}
    train_ds = CyberBullyDataset(train_df, tokenizer, config['max_length'],
                                  active_tasks, label_encoders, text_col)
    val_ds   = CyberBullyDataset(val_df,   tokenizer, config['max_length'],
                                  active_tasks, label_encoders, text_col)
    test_ds  = CyberBullyDataset(test_df,  tokenizer, config['max_length'],
                                  active_tasks, label_encoders, text_col)
    train_dl = DataLoader(train_ds, batch_size=config['batch_size'],
                           shuffle=True,  num_workers=0, pin_memory=True)
    val_dl   = DataLoader(val_ds,   batch_size=config['batch_size'],
                           shuffle=False, num_workers=0, pin_memory=True)
    test_dl  = DataLoader(test_ds,  batch_size=config['batch_size'],
                           shuffle=False, num_workers=0, pin_memory=True)

    model = MultiTaskTransformer(BASE_MODEL, active_tasks).to(device)

    # Loss functions
    criteria = {}
    for t, cfg in active_tasks.items():
        col = cfg['col']
        enc = label_encoders[t]
        n   = cfg['num_classes']
        cc  = train_df[col].map(enc).value_counts().sort_index()
        w   = np.ones(n, dtype=np.float32)
        for idx, cnt in cc.items():
            if cnt > 0: w[int(idx)] = 1.0 / cnt
        w = w / w.sum() * n
        wt = torch.tensor(w, dtype=torch.float32).to(device)
        if use_focal:
            criteria[t] = FocalLoss(gamma=config['focal_gamma'], weight=wt)
        else:
            criteria[t] = nn.CrossEntropyLoss(weight=wt)

    # Optimizer
    if use_lrdecay:
        param_groups = get_layerwise_params(model, config['learning_rate'],
                                             config['lr_decay_factor'],
                                             config['weight_decay'])
    else:
        # Uniform LR — no decay
        no_decay = ['bias', 'LayerNorm.weight', 'LayerNorm.bias']
        param_groups = [
            {'params': [p for n,p in model.named_parameters()
                        if not any(nd in n for nd in no_decay)],
             'lr': config['learning_rate'], 'weight_decay': config['weight_decay']},
            {'params': [p for n,p in model.named_parameters()
                        if any(nd in n for nd in no_decay)],
             'lr': config['learning_rate'], 'weight_decay': 0.0},
        ]
    optimizer = torch.optim.AdamW(param_groups)
    total_steps   = len(train_dl) * config['epochs']
    warmup_steps  = int(total_steps * config['warmup_ratio'])
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    scaler = torch.cuda.amp.GradScaler() if config['use_fp16'] else None

    save_dir = f'../outputs/ablations/{tag}'
    os.makedirs(save_dir, exist_ok=True)
    best_val_f1 = 0; patience_counter = 0

    for epoch in range(config['epochs']):
        model.train(); total_loss = 0
        for batch in train_dl:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad()
            with torch.autocast(device_type='cuda', enabled=config['use_fp16']):
                logits = model(input_ids=batch['input_ids'],
                               attention_mask=batch['attention_mask'],
                               token_type_ids=batch.get('token_type_ids'))
                loss = 0
                for t, cfg in active_tasks.items():
                    lbl  = batch[f'label_{t}']
                    mask = lbl >= 0
                    if mask.sum() > 0:
                        loss += cfg['loss_weight'] * criteria[t](logits[t][mask], lbl[mask])
            if scaler:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            scheduler.step(); total_loss += loss.item()

        val_m = evaluate_binary(model, val_dl)
        f1 = val_m['macro_f1']
        status = ''
        if f1 > best_val_f1:
            best_val_f1 = f1
            torch.save(model.state_dict(), f'{save_dir}/best_model.pt')
            patience_counter = 0; status = ' BEST'
        else:
            patience_counter += 1
        print(f'  Epoch {epoch+1:2d}/{config["epochs"]} | '
              f'Loss: {total_loss/len(train_dl):.4f} | '
              f'Val Macro-F1: {f1:.4f}{status}')
        if patience_counter >= config['patience']:
            print(f'  Early stopping at epoch {epoch+1}'); break

    model.load_state_dict(torch.load(f'{save_dir}/best_model.pt', map_location=device, weights_only=True))
    test_m = evaluate_binary(model, test_dl)
    result = {'tag': tag, 'best_val_f1': round(best_val_f1, 4), **test_m}
    with open(f'{save_dir}/results.json','w') as fh: json.dump(result, fh, indent=2)
    print(f'  Test Macro-F1: {test_m["macro_f1"]:.4f} | Accuracy: {test_m["accuracy"]:.4f}')
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return result

print('Training function ready')


Training function ready


## Run all ablations

In [6]:
import copy
ablation_results = []

# 1. Full multi-task (reference baseline for ablation table)
cfg = copy.deepcopy(full_cfg)
enc = build_label_encoders(cfg)
r = run_ablation('full_multitask', cfg, enc, BASE_CONFIG,
                  use_focal=True, use_lrdecay=True, text_col='text_clean')
ablation_results.append(r)
pd.DataFrame(ablation_results).to_csv('../outputs/ablations/ablation_results.csv', index=False)

# 2. Binary-only (no abuse_type head)
cfg = copy.deepcopy(bin_cfg)
enc = build_label_encoders(cfg)
r = run_ablation('binary_only', cfg, enc, BASE_CONFIG,
                  use_focal=True, use_lrdecay=True, text_col='text_clean')
ablation_results.append(r)
pd.DataFrame(ablation_results).to_csv('../outputs/ablations/ablation_results.csv', index=False)

# 3. No focal loss (standard cross-entropy with class weights)
cfg = copy.deepcopy(full_cfg)
enc = build_label_encoders(cfg)
r = run_ablation('no_focal', cfg, enc, BASE_CONFIG,
                  use_focal=False, use_lrdecay=True, text_col='text_clean')
ablation_results.append(r)
pd.DataFrame(ablation_results).to_csv('../outputs/ablations/ablation_results.csv', index=False)

# 4. No layer-wise LR decay (uniform LR)
cfg = copy.deepcopy(full_cfg)
enc = build_label_encoders(cfg)
r = run_ablation('no_lrdecay', cfg, enc, BASE_CONFIG,
                  use_focal=True, use_lrdecay=False, text_col='text_clean')
ablation_results.append(r)
pd.DataFrame(ablation_results).to_csv('../outputs/ablations/ablation_results.csv', index=False)

# 5. No preprocessing (raw text)
cfg = copy.deepcopy(full_cfg)
enc = build_label_encoders(cfg)
r = run_ablation('no_preprocessing', cfg, enc, BASE_CONFIG,
                  use_focal=True, use_lrdecay=True, text_col='text')
ablation_results.append(r)
pd.DataFrame(ablation_results).to_csv('../outputs/ablations/ablation_results.csv', index=False)


  Task 'binary': 2 classes
  Task 'abuse_type': 9 classes

ABLATION: full_multitask  |  banglabert  |  seed=42


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 33497.30it/s]
ElectraModel LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
electra.embeddings.position_ids                   | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


KeyboardInterrupt: 

## Summary

In [ ]:
abl_df = pd.DataFrame(ablation_results)
print('=' * 70)
print('ABLATION RESULTS (BanglaBERT, seed=42)')
print('=' * 70)
print(abl_df[['tag','macro_f1','accuracy','mcc']].to_string(index=False))

# Compute deltas vs full_multitask
ref = abl_df.loc[abl_df['tag']=='full_multitask','macro_f1'].values[0]
abl_df['delta'] = (abl_df['macro_f1'] - ref).round(4)
print(f'\nReference (full_multitask): Macro-F1 = {ref:.4f}')
print(abl_df[['tag','macro_f1','delta']].to_string(index=False))
print('\nSaved: ../outputs/ablations/ablation_results.csv')
